# M2: Ventricular-tachycardia alarm baseline

Run cells in order in Google Colab with a GPU runtime. Inputs and checkpoints are stored in the author's Drive; adjust `RUN_DIR` and source archive paths for another environment. Cells 1–4 prepare, train, and verify five fixed seeds; Cells 5–6 freeze the plan and evaluate the official held-out test once. Do not select models or thresholds using test results.


In [ ]:
# M2 CELL 1 — restore verified development data; no downloads or training.
# Initial ETA ~5–30 seconds after Drive authorization; measured hash progress follows.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, time, platform, importlib.metadata
import numpy as np
import torch

RUN_DIR = Path('/content/drive/MyDrive/SentinelWard-work/data-audit/20261015T132438_754996Z')
PREP_DIR = RUN_DIR / 'vtac_m2_prepared_v1'
path = PREP_DIR / 'development.npz'
print('Verifying saved M2 inputs — initial ETA ~5–30s.', flush=True)
started = time.monotonic()
digest = hashlib.sha256()
total = path.stat().st_size
done = 0
with path.open('rb') as stream:
    for block in iter(lambda: stream.read(8 * 1024**2), b''):
        digest.update(block)
        done += len(block)
        elapsed = time.monotonic() - started
        print(f'Hash {done/total:.0%} | elapsed {elapsed:.1f}s | ETA {elapsed/done*(total-done):.1f}s', flush=True)
assert digest.hexdigest() == (PREP_DIR/'development.sha256').read_text().strip(), 'Prepared data checksum mismatch'
preprocessing = json.loads((PREP_DIR/'preprocessing.json').read_text())
assert preprocessing['channel_order'] == ['ecg_z','ppg_z','ecg_valid','ppg_valid']
assert preprocessing['input_shape'] == [4,1250]
assert preprocessing['interval_seconds'] == [290,300]
with np.load(path, allow_pickle=False) as data:
    X, y = data['X'].copy(), data['y'].copy()
    events, records, splits = (data[k].copy() for k in ('event_id','record_id','split'))
assert X.shape == (4555,4,1250) and np.isfinite(X).all()
assert y.shape == events.shape == records.shape == splits.shape == (4555,)
assert len(set(events.tolist())) == 4555 and set(np.unique(y)) == {0,1}
assert set(splits.tolist()) == {'train','validation'}
train_idx, val_idx = np.flatnonzero(splits=='train'), np.flatnonzero(splits=='validation')
assert len(train_idx)==4060 and len(val_idx)==495
assert int(y[train_idx].sum())==1163 and int(y[val_idx].sum())==141
assert not set(records[train_idx]) & set(records[val_idx])
assert np.isin(X[:,2:,:], [0,1]).all()
assert (X[:,:2,:][X[:,2:,:]==0] == 0).all()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PASS: inputs, masks, labels and record-disjoint splits; test events loaded: 0')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('Versions:', {p:importlib.metadata.version(p) for p in ('numpy','scipy','torch','scikit-learn')})
print(f'Completed in {time.monotonic()-started:.1f}s. M2 training has NOT started.')


In [ ]:
# M2 CELL 2 — train one compact VT-alarm model.
# GPU ETA: approximately 2–8 minutes. Prints progress after every epoch.

from pathlib import Path
from datetime import datetime, timezone
import copy, hashlib, json, random, time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)

SEED = 20261015
MAX_EPOCHS = 30
PATIENCE = 6
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
RECALL_TARGET = 0.90

assert "X" in globals(), "Run M2 Cell 1 again after changing runtime."
assert device.type == "cuda", (
    "GPU is not active. Select Runtime > Change runtime type > T4 GPU, "
    "then rerun Cell 1."
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

train_idx = np.flatnonzero(splits == "train")
val_idx = np.flatnonzero(splits == "validation")

assert len(train_idx) == 4060
assert len(val_idx) == 495
assert not set(records[train_idx]) & set(records[val_idx])

X_train = torch.from_numpy(X[train_idx]).float()
y_train = torch.from_numpy(y[train_idx]).float()
X_val = torch.from_numpy(X[val_idx]).float()
y_val = torch.from_numpy(y[val_idx]).float()

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
    num_workers=0,
    pin_memory=True,
)
val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)


class CompactM2FCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(4, 32, 15, padding=7, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, 9, padding=4, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 96, 5, padding=2, bias=False),
            nn.BatchNorm1d(96),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.25),
            nn.Linear(96, 1),
        )

    def forward(self, values):
        return self.classifier(self.features(values)).squeeze(1)


model = CompactM2FCN().to(device)

positive = int(y_train.sum().item())
negative = len(y_train) - positive
pos_weight = torch.tensor([negative / positive], device=device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=1e-5,
)
scaler = torch.amp.GradScaler("cuda", enabled=True)


@torch.no_grad()
def predict(loader):
    model.eval()
    labels, probabilities = [], []

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=True):
            logits = model(batch_x)

        probabilities.append(torch.sigmoid(logits).cpu().numpy())
        labels.append(batch_y.numpy())

    return (
        np.concatenate(labels).astype(int),
        np.concatenate(probabilities),
    )


print("Training M2 compact FCN")
print("GPU:", torch.cuda.get_device_name(0))
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Train events:", len(train_idx), "| true VT:", positive)
print("Validation events:", len(val_idx), "| true VT:", int(y_val.sum()))
print("Test events loaded: 0")
print("Initial GPU ETA: approximately 2–8 minutes.", flush=True)

started = time.monotonic()
best_ap = -np.inf
best_epoch = None
best_state = None
epochs_without_improvement = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    seen = 0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=True):
            logits = model(batch_x)
            loss = criterion(logits, batch_y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * len(batch_y)
        seen += len(batch_y)

    val_true, val_probability = predict(val_loader)
    val_ap = average_precision_score(val_true, val_probability)
    val_auroc = roc_auc_score(val_true, val_probability)
    train_loss = running_loss / seen
    scheduler.step(val_ap)

    improved = val_ap > best_ap + 1e-5
    if improved:
        best_ap = float(val_ap)
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    elapsed = time.monotonic() - started
    planned_epochs = min(
        MAX_EPOCHS,
        epoch + max(0, PATIENCE - epochs_without_improvement),
    )
    eta = (elapsed / epoch) * max(0, planned_epochs - epoch)

    history.append({
        "epoch": epoch,
        "train_loss": float(train_loss),
        "validation_average_precision": float(val_ap),
        "validation_auroc": float(val_auroc),
        "learning_rate": float(optimizer.param_groups[0]["lr"]),
        "best": bool(improved),
    })

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"loss {train_loss:.4f} | "
        f"val AP {val_ap:.4f} | "
        f"val AUROC {val_auroc:.4f} | "
        f"elapsed {elapsed:.1f}s | ETA {eta:.1f}s"
        + (" | BEST" if improved else ""),
        flush=True,
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after {PATIENCE} epochs without improvement.")
        break

assert best_state is not None
model.load_state_dict(best_state)
val_true, val_probability = predict(val_loader)

# Select the threshold from validation data only.
threshold_rows = []
candidates = np.unique(
    np.concatenate(([0.0], val_probability, [1.0]))
)

for candidate in candidates:
    prediction = (val_probability >= candidate).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        val_true, prediction, labels=[0, 1]
    ).ravel()

    threshold_rows.append({
        "threshold": float(candidate),
        "recall": float(tp / (tp + fn)),
        "specificity": float(tn / (tn + fp)),
        "precision": float(tp / (tp + fp)) if tp + fp else 0.0,
    })

eligible = [
    row for row in threshold_rows
    if row["recall"] >= RECALL_TARGET
]
assert eligible

selected = max(
    eligible,
    key=lambda row: (
        row["specificity"],
        row["precision"],
        row["threshold"],
    ),
)
threshold = selected["threshold"]
val_prediction = (val_probability >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(
    val_true, val_prediction, labels=[0, 1]
).ravel()

metrics = {
    "best_epoch": int(best_epoch),
    "threshold": float(threshold),
    "threshold_selection":
        f"Validation recall >= {RECALL_TARGET:.2f}; maximize specificity",
    "n": int(len(val_true)),
    "true_vt": int(val_true.sum()),
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "TP": int(tp),
    "recall_true_vt": float(recall_score(val_true, val_prediction)),
    "specificity_false_alarm": float(tn / (tn + fp)),
    "precision_true_vt": float(
        precision_score(val_true, val_prediction, zero_division=0)
    ),
    "f1_true_vt": float(f1_score(val_true, val_prediction)),
    "auroc": float(roc_auc_score(val_true, val_probability)),
    "average_precision": float(
        average_precision_score(val_true, val_probability)
    ),
}

output_dir = (
    RUN_DIR
    / "m2_training"
    / datetime.now(timezone.utc).strftime(
        "m2_fcn_seed20261015_%Y%m%dT%H%M%S_%fZ"
    )
)
output_dir.mkdir(parents=True, exist_ok=False)

configuration = {
    "model": "CompactM2FCN",
    "seed": SEED,
    "input_channels": [
        "ecg_z", "ppg_z", "ecg_valid", "ppg_valid"
    ],
    "input_shape": [4, 1250],
    "sampling_rate_hz": 125,
    "window_seconds": 10,
    "batch_size": BATCH_SIZE,
    "maximum_epochs": MAX_EPOCHS,
    "early_stopping_patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "positive_class_weight": float(pos_weight.item()),
    "selection_metric": "validation average precision",
    "recall_target": RECALL_TARGET,
    "test_events_processed": 0,
    "torch_version": torch.__version__,
}

checkpoint_path = output_dir / "best_model.pt"
torch.save({
    "model_state_dict": best_state,
    "configuration": configuration,
    "validation_metrics": metrics,
}, checkpoint_path)

pd.DataFrame(history).to_csv(
    output_dir / "training_history.csv", index=False
)
pd.DataFrame({
    "event_id": events[val_idx],
    "record_id": records[val_idx],
    "target_true_vt": val_true,
    "probability_true_vt": val_probability,
    "prediction": val_prediction,
}).to_csv(
    output_dir / "validation_predictions.csv", index=False
)

(output_dir / "configuration.json").write_text(
    json.dumps(configuration, indent=2), encoding="utf-8"
)
(output_dir / "validation_metrics.json").write_text(
    json.dumps(metrics, indent=2), encoding="utf-8"
)

checkpoint_sha256 = hashlib.sha256(
    checkpoint_path.read_bytes()
).hexdigest()
(output_dir / "best_model.sha256").write_text(
    checkpoint_sha256, encoding="utf-8"
)

print("\nM2 VALIDATION RESULT")
print(json.dumps(metrics, indent=2))
print("\nCheckpoint SHA-256:", checkpoint_sha256)
print("Saved:", output_dir)
print("Test events processed: 0")
print(f"Completed in {time.monotonic() - started:.1f}s.")

In [ ]:
# M2 CELL 3 — four additional fixed seeds; five-seed validation summary.
# GPU ETA: ~1–3 minutes. Test events remain untouched.

from datetime import datetime, timezone
from pathlib import Path
import copy, hashlib, json, random, time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)

assert device.type == "cuda"
assert "CompactM2FCN" in globals(), "Run M2 Cell 2 first."
assert output_dir.is_dir(), "The first M2 checkpoint folder is missing."

first_checkpoint = output_dir / "best_model.pt"
first_hash = hashlib.sha256(first_checkpoint.read_bytes()).hexdigest()
assert first_hash == (output_dir / "best_model.sha256").read_text().strip()

first_metrics = json.loads(
    (output_dir / "validation_metrics.json").read_text()
)
first_predictions = pd.read_csv(
    output_dir / "validation_predictions.csv",
    dtype={"event_id": str, "record_id": str},
)
assert first_metrics["best_epoch"] == 24
assert first_metrics["n"] == 495
assert first_predictions["event_id"].tolist() == events[val_idx].astype(str).tolist()
assert first_predictions["target_true_vt"].to_numpy().tolist() == y[val_idx].tolist()

run_dir = (
    output_dir.parent
    / datetime.now(timezone.utc).strftime(
        "m2_five_seed_%Y%m%dT%H%M%S_%fZ"
    )
)
run_dir.mkdir(parents=True, exist_ok=False)

val_labels = y[val_idx].astype(int)
ppg_available = X[val_idx, 3, :].any(axis=1)
results = []
subgroups = []


def choose_threshold(labels, probabilities):
    """Meet the predeclared recall target, then maximize specificity."""
    candidates = np.unique(
        np.concatenate(([0.0], probabilities, [1.0]))
    )
    eligible = []

    for candidate in candidates:
        predicted = (probabilities >= candidate).astype(int)
        tn, fp, fn, tp = confusion_matrix(
            labels, predicted, labels=[0, 1]
        ).ravel()
        recall = tp / (tp + fn)
        if recall >= RECALL_TARGET:
            eligible.append((
                tn / (tn + fp),
                tp / (tp + fp) if tp + fp else 0.0,
                float(candidate),
            ))

    assert eligible
    return max(eligible)[2]


def evaluate(labels, probabilities, threshold):
    predicted = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        labels, predicted, labels=[0, 1]
    ).ravel()

    return {
        "n": int(len(labels)),
        "true_vt": int(labels.sum()),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "recall_true_vt": float(
            recall_score(labels, predicted, zero_division=0)
        ),
        "specificity_false_alarm": float(
            tn / (tn + fp) if tn + fp else float("nan")
        ),
        "precision_true_vt": float(
            precision_score(labels, predicted, zero_division=0)
        ),
        "f1_true_vt": float(
            f1_score(labels, predicted, zero_division=0)
        ),
        "auroc": float(roc_auc_score(labels, probabilities))
            if len(np.unique(labels)) == 2 else float("nan"),
        "average_precision": float(
            average_precision_score(labels, probabilities)
        ) if len(np.unique(labels)) == 2 else float("nan"),
    }


def record_seed(seed, best_epoch, threshold, probabilities):
    metrics = evaluate(val_labels, probabilities, threshold)
    metrics.update({
        "seed": int(seed),
        "best_epoch": int(best_epoch),
        "threshold": float(threshold),
    })
    results.append(metrics)

    for name, mask in (
        ("ppg_available", ppg_available),
        ("ppg_unavailable", ~ppg_available),
    ):
        subgroup_metrics = evaluate(
            val_labels[mask],
            probabilities[mask],
            threshold,
        )
        subgroup_metrics.update({"seed": int(seed), "scope": name})
        subgroups.append(subgroup_metrics)

    pd.DataFrame({
        "event_id": events[val_idx],
        "record_id": records[val_idx],
        "target_true_vt": val_labels,
        "probability_true_vt": probabilities,
        "prediction": (probabilities >= threshold).astype(int),
        "ppg_available": ppg_available,
    }).to_csv(
        run_dir / f"validation_predictions_seed{seed}.csv",
        index=False,
    )


# Reuse the already trained and saved seed 20261015.
record_seed(
    SEED,
    first_metrics["best_epoch"],
    first_metrics["threshold"],
    first_predictions["probability_true_vt"].to_numpy(),
)

started = time.monotonic()
remaining_seeds = [20261016, 20261017, 20261018, 20261019]

print("Training four additional fixed seeds — initial GPU ETA ~1–3 minutes.")
print("Existing seed verified:", SEED)
print("Test events loaded: 0", flush=True)

for seed_number, seed in enumerate(remaining_seeds, start=1):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=True,
    )

    current_model = CompactM2FCN().to(device)
    current_optimizer = torch.optim.AdamW(
        current_model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    current_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        current_optimizer,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
    )
    current_scaler = torch.amp.GradScaler("cuda", enabled=True)

    best_ap = -np.inf
    best_epoch = None
    best_state = None
    stale_epochs = 0
    seed_history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        current_model.train()
        loss_total = 0.0
        seen = 0

        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)

            current_optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=True):
                logits = current_model(batch_x)
                loss = criterion(logits, batch_y)

            current_scaler.scale(loss).backward()
            current_scaler.step(current_optimizer)
            current_scaler.update()

            loss_total += loss.item() * len(batch_y)
            seen += len(batch_y)

        current_model.eval()
        probabilities = []
        with torch.no_grad():
            for batch_x, _ in val_loader:
                batch_x = batch_x.to(device, non_blocking=True)
                with torch.amp.autocast("cuda", enabled=True):
                    logits = current_model(batch_x)
                probabilities.append(torch.sigmoid(logits).cpu().numpy())

        probabilities = np.concatenate(probabilities)
        val_ap = average_precision_score(val_labels, probabilities)
        current_scheduler.step(val_ap)

        improved = val_ap > best_ap + 1e-5
        if improved:
            best_ap = float(val_ap)
            best_epoch = epoch
            best_state = copy.deepcopy(current_model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1

        seed_history.append({
            "seed": seed,
            "epoch": epoch,
            "train_loss": float(loss_total / seen),
            "validation_average_precision": float(val_ap),
            "best": bool(improved),
        })

        elapsed = time.monotonic() - started
        completed_fraction = (
            (seed_number - 1) + epoch / MAX_EPOCHS
        ) / len(remaining_seeds)
        eta = (
            elapsed / completed_fraction * (1 - completed_fraction)
            if completed_fraction else float("nan")
        )
        print(
            f"Seed {seed} | epoch {epoch:02d}/{MAX_EPOCHS} | "
            f"val AP {val_ap:.4f} | elapsed {elapsed:.1f}s | "
            f"rough ETA {eta:.1f}s"
            + (" | BEST" if improved else ""),
            flush=True,
        )

        if stale_epochs >= PATIENCE:
            break

    assert best_state is not None
    current_model.load_state_dict(best_state)
    current_model.eval()

    probabilities = []
    with torch.no_grad():
        for batch_x, _ in val_loader:
            batch_x = batch_x.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=True):
                logits = current_model(batch_x)
            probabilities.append(torch.sigmoid(logits).cpu().numpy())
    probabilities = np.concatenate(probabilities)

    threshold = choose_threshold(val_labels, probabilities)
    record_seed(seed, best_epoch, threshold, probabilities)

    checkpoint = run_dir / f"best_model_seed{seed}.pt"
    torch.save({
        "model_state_dict": best_state,
        "seed": seed,
        "best_epoch": best_epoch,
        "threshold": threshold,
        "architecture": "CompactM2FCN",
        "source_preprocessing": preprocessing,
    }, checkpoint)
    (run_dir / f"best_model_seed{seed}.sha256").write_text(
        hashlib.sha256(checkpoint.read_bytes()).hexdigest(),
        encoding="utf-8",
    )
    pd.DataFrame(seed_history).to_csv(
        run_dir / f"history_seed{seed}.csv",
        index=False,
    )

    print(
        f"Seed {seed} complete | best epoch {best_epoch} | "
        f"threshold {threshold:.4f}",
        flush=True,
    )

results_df = pd.DataFrame(results).sort_values("seed")
subgroups_df = pd.DataFrame(subgroups).sort_values(["scope", "seed"])

metric_columns = [
    "recall_true_vt",
    "specificity_false_alarm",
    "precision_true_vt",
    "f1_true_vt",
    "auroc",
    "average_precision",
]
summary = results_df[metric_columns].agg(["mean", "std"])

results_df.to_csv(run_dir / "five_seed_validation_metrics.csv", index=False)
subgroups_df.to_csv(run_dir / "five_seed_ppg_subgroups.csv", index=False)
summary.to_csv(run_dir / "five_seed_summary.csv")

(run_dir / "run_manifest.json").write_text(
    json.dumps({
        "seeds": [SEED] + remaining_seeds,
        "first_checkpoint": str(first_checkpoint),
        "first_checkpoint_sha256": first_hash,
        "threshold_rule":
            f"validation recall >= {RECALL_TARGET}; maximize specificity",
        "model_selection": "best validation average precision",
        "source_development_sha256": (
            PREP_DIR / "development.sha256"
        ).read_text().strip(),
        "test_events_processed": 0,
    }, indent=2),
    encoding="utf-8",
)

print("\nFIVE-SEED VALIDATION RESULTS")
print(results_df[
    ["seed", "best_epoch", "threshold", "TN", "FP", "FN", "TP"]
    + metric_columns
].round(4).to_string(index=False))

print("\nMEAN AND SAMPLE SD ACROSS SEEDS")
print(summary.round(4).to_string())

print("\nPPG AVAILABILITY SUBGROUPS")
print(subgroups_df.groupby("scope")[
    ["n", "true_vt", "recall_true_vt",
     "specificity_false_alarm", "average_precision"]
].mean().round(4).to_string())

print("\nSaved:", run_dir)
print("Test events processed: 0")
print(f"Completed in {time.monotonic() - started:.1f}s.")

In [ ]:
# M2 CELL 4 — correct and verify five-seed validation summaries.
# ETA: a few seconds. No training or test access.

import json
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix

assert "run_dir" in globals(), "Run M2 Cell 3 first."
assert "output_dir" in globals(), "Run M2 Cell 2 first."

seeds = [20261015, 20261016, 20261017, 20261018, 20261019]
original = pd.read_csv(run_dir / "five_seed_validation_metrics.csv")
assert original["seed"].tolist() == seeds

first_metrics = json.loads(
    (output_dir / "validation_metrics.json").read_text()
)
corrected = original.copy()

# Cell 3 recalculated seed 1's binary predictions from rounded CSV scores.
# Restore its original metrics from Cell 2's in-memory evaluation.
first_row = corrected["seed"] == 20261015
for column in (
    "TN", "FP", "FN", "TP",
    "recall_true_vt", "specificity_false_alarm",
    "precision_true_vt", "f1_true_vt",
    "auroc", "average_precision",
):
    corrected.loc[first_row, column] = first_metrics[column]

ppg_available = X[val_idx, 3, :].any(axis=1)
subgroup_rows = []

for seed in seeds:
    # For seed 1, use the ORIGINAL prediction file written by Cell 2.
    # For seeds 2–5, use the files written by Cell 3.
    prediction_path = (
        output_dir / "validation_predictions.csv"
        if seed == 20261015
        else run_dir / f"validation_predictions_seed{seed}.csv"
    )
    saved = pd.read_csv(
        prediction_path,
        dtype={"event_id": str, "record_id": str},
    )

    assert saved["event_id"].tolist() == events[val_idx].astype(str).tolist()
    truth = saved["target_true_vt"].to_numpy(dtype=int)
    predicted = saved["prediction"].to_numpy(dtype=int)
    probability = saved["probability_true_vt"].to_numpy(dtype=float)
    np.testing.assert_array_equal(truth, y[val_idx])

    counts = np.array(
        confusion_matrix(truth, predicted, labels=[0, 1]).ravel()
    )
    expected = corrected.loc[
        corrected["seed"] == seed,
        ["TN", "FP", "FN", "TP"],
    ].iloc[0].to_numpy(dtype=int)

    print(
        f"Seed {seed}: saved TN/FP/FN/TP "
        f"{counts.tolist()} | expected {expected.tolist()}"
    )
    np.testing.assert_array_equal(
        counts, expected,
        err_msg=f"Original saved predictions disagree for seed {seed}",
    )

    for scope, mask in (
        ("ppg_available", ppg_available),
        ("ppg_unavailable", ~ppg_available),
    ):
        part_truth = truth[mask]
        part_predicted = predicted[mask]
        part_probability = probability[mask]
        tn, fp, fn, tp = confusion_matrix(
            part_truth, part_predicted, labels=[0, 1]
        ).ravel()

        subgroup_rows.append({
            "seed": seed,
            "scope": scope,
            "n": int(mask.sum()),
            "true_vt": int(part_truth.sum()),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "TP": int(tp),
            "recall_true_vt": (
                float(tp / (tp + fn)) if tp + fn else float("nan")
            ),
            "specificity_false_alarm": (
                float(tn / (tn + fp)) if tn + fp else float("nan")
            ),
            "average_precision": (
                float(average_precision_score(
                    part_truth, part_probability
                ))
                if len(np.unique(part_truth)) == 2
                else float("nan")
            ),
        })

metric_columns = [
    "recall_true_vt",
    "specificity_false_alarm",
    "precision_true_vt",
    "f1_true_vt",
    "auroc",
    "average_precision",
]
summary = corrected[metric_columns].agg(["mean", "std"])
subgroups = pd.DataFrame(subgroup_rows)

correction_dir = run_dir / "corrected_validation"
correction_dir.mkdir(exist_ok=True)

corrected.to_csv(
    correction_dir / "five_seed_validation_metrics.csv",
    index=False,
)
summary.to_csv(
    correction_dir / "five_seed_summary.csv",
)
subgroups.to_csv(
    correction_dir / "five_seed_ppg_subgroups.csv",
    index=False,
)

(correction_dir / "correction_note.json").write_text(
    json.dumps({
        "reason": (
            "Cell 3 reconstructed seed 20261015 predictions from "
            "rounded probability text. This correction uses the "
            "original Cell 2 prediction file and metrics."
        ),
        "seed_20261015_prediction_source":
            str(output_dir / "validation_predictions.csv"),
        "other_prediction_source": str(run_dir),
        "all_five_saved_prediction_counts_verified": True,
        "test_events_processed": 0,
    }, indent=2),
    encoding="utf-8",
)

print("\nCORRECTED FIVE-SEED VALIDATION")
print(corrected[
    ["seed", "best_epoch", "threshold", "TN", "FP", "FN", "TP"]
    + metric_columns
].round(4).to_string(index=False))

print("\nCORRECTED MEAN AND SAMPLE SD")
print(summary.round(4).to_string())

print("\nPPG SUBGROUPS — MEAN ACROSS SEEDS")
print(subgroups.groupby("scope")[
    ["n", "true_vt", "recall_true_vt",
     "specificity_false_alarm", "average_precision"]
].mean().round(4).to_string())

print("\nSaved:", correction_dir)
print("Test events processed: 0")

In [ ]:
# M2 CELL 5 — freeze models and prepare the official VTaC test inputs.
# Initial ETA: ~2–10 minutes, depending on Google Drive speed.
# No fitting, threshold changes, or model scoring.
%pip -q install --no-deps wfdb==4.3.1
from pathlib import Path, PurePosixPath
import hashlib, io, json, tempfile, time, zipfile

import numpy as np
import pandas as pd
import scipy
from scipy.signal import firwin, resample_poly
from scipy.ndimage import minimum_filter1d
import wfdb

assert "run_dir" in globals() and "output_dir" in globals()
assert "RUN_DIR" in globals() and "PREP_DIR" in globals()

started = time.monotonic()
print("Freezing five models and preparing test data — ETA ~2–10 min.",
      flush=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()

# Freeze model identities and thresholds before reading test waveforms.
corrected_path = (
    run_dir / "corrected_validation" / "five_seed_validation_metrics.csv"
)
selection = pd.read_csv(corrected_path).sort_values("seed")
seeds = [20261015, 20261016, 20261017, 20261018, 20261019]
assert selection["seed"].tolist() == seeds

frozen_models = []
for row in selection.itertuples(index=False):
    checkpoint = (
        output_dir / "best_model.pt"
        if row.seed == 20261015
        else run_dir / f"best_model_seed{row.seed}.pt"
    )
    checksum_file = (
        output_dir / "best_model.sha256"
        if row.seed == 20261015
        else run_dir / f"best_model_seed{row.seed}.sha256"
    )
    actual = sha256_file(checkpoint)
    assert actual == checksum_file.read_text().strip()
    frozen_models.append({
        "seed": int(row.seed),
        "checkpoint": str(checkpoint),
        "checkpoint_sha256": actual,
        "best_epoch": int(row.best_epoch),
        "threshold": float(row.threshold),
    })

frozen_plan = {
    "models": frozen_models,
    "preprocessing_sha256": sha256_file(
        PREP_DIR / "preprocessing.json"
    ),
    "development_npz_sha256": (
        PREP_DIR / "development.sha256"
    ).read_text().strip(),
    "selection": "Best validation AP; validation recall >= 0.90, "
                 "then maximum specificity",
    "test_use": "One frozen evaluation; no further model selection",
}
frozen_path = run_dir / "frozen_test_plan.json"
if frozen_path.exists():
    assert json.loads(frozen_path.read_text()) == frozen_plan
else:
    frozen_path.write_text(
        json.dumps(frozen_plan, indent=2), encoding="utf-8"
    )

source_zip = (
    Path("/content/drive/MyDrive/SentinelWard-data/vtac-1.1")
    / "vtac-a-benchmark-dataset-of-ventricular-tachycardia-alarms-from-icu-monitors-1.1.zip"
)
assert source_zip.is_file(), f"VTaC ZIP missing: {source_zip}"
assert source_zip.stat().st_size == 11695857538

prior_provenance = json.loads(
    (RUN_DIR / "vtac_header_manifest_provenance.json").read_text()
)
development_manifest = pd.read_csv(
    RUN_DIR / "vtac_development_header_manifest.csv",
    dtype={"record_id": str, "event_id": str},
)

assert preprocessing["channel_order"] == [
    "ecg_z", "ppg_z", "ecg_valid", "ppg_valid"
]
assert preprocessing["interval_seconds"] == [290, 300]
assert preprocessing["scipy_version"] == scipy.__version__
assert preprocessing["numpy_version"] == np.__version__

test_dir = RUN_DIR / "vtac_m2_test_prepared_v1"
test_dir.mkdir(parents=True, exist_ok=True)
batch_size = 50
ecg_preference = [
    "II", "V", "I", "III", "aVR", "aVL", "aVF",
    "V1", "V2", "V3", "V4", "V5", "V6",
]

with zipfile.ZipFile(source_zip) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names))
    name_set = set(names)

    def unique_name(basename):
        matches = [
            name for name in names
            if PurePosixPath(name).name == basename
        ]
        assert len(matches) == 1, f"Missing or ambiguous {basename}"
        return matches[0]

    checksum_member = unique_name("SHA256SUMS.txt")
    archive_root = str(PurePosixPath(checksum_member).parent)
    checksum_bytes = archive.read(checksum_member)

    assert hashlib.sha256(checksum_bytes).hexdigest() == (
        prior_provenance["publisher_checksum_manifest_sha256"]
    )

    publisher_hashes = {}
    for line in checksum_bytes.decode("utf-8-sig").splitlines():
        if not line.strip():
            continue
        digest, name = line.split(maxsplit=1)
        name = name.lstrip("*")
        if name.startswith("./"):
            name = name[2:]
        assert name not in publisher_hashes
        publisher_hashes[name] = digest.lower()

    def verified_read(member):
        relative = str(
            PurePosixPath(member).relative_to(archive_root)
        )
        matches = {
            publisher_hashes[key]
            for key in (relative, member)
            if key in publisher_hashes
        }
        assert len(matches) == 1, (
            f"No unique publisher checksum for {member}"
        )
        payload = archive.read(member)
        assert hashlib.sha256(payload).hexdigest() == next(iter(matches)), (
            f"Publisher checksum failed: {member}"
        )
        return payload

    labels_bytes = verified_read(unique_name("event_labels.csv"))
    splits_bytes = verified_read(
        unique_name("benchmark_data_split.csv")
    )

    assert hashlib.sha256(labels_bytes).hexdigest() == (
        prior_provenance["labels_sha256"]
    )
    assert hashlib.sha256(splits_bytes).hexdigest() == (
        prior_provenance["official_split_sha256"]
    )

    labels = pd.read_csv(
        io.BytesIO(labels_bytes), dtype=str, keep_default_na=False
    )
    split_table = pd.read_csv(
        io.BytesIO(splits_bytes), dtype=str, keep_default_na=False
    )
    joined = labels.merge(
        split_table,
        on=["record", "event"],
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
    assert len(joined) == 5037
    assert joined["_merge"].eq("both").all()
    assert set(joined["decision"]) == {"True", "False"}

    test_rows = joined[
        joined["split"].str.strip().str.lower().eq("test")
    ].sort_values(["record", "event"]).reset_index(drop=True)

    assert len(test_rows) == 482
    assert test_rows["record"].nunique() == 226
    assert int(test_rows["decision"].eq("True").sum()) == 137
    assert not set(test_rows["record"]) & set(
        development_manifest["record_id"]
    )
    assert test_rows["event"].is_unique

    source_id = hashlib.sha256(
        labels_bytes + splits_bytes + checksum_bytes
    ).hexdigest()
    policy = {
        "version": "m2-test-preparation-v1",
        "source_id": source_id,
        "interval_seconds": [290, 300],
        "native_sampling_rate_hz": 250,
        "model_sampling_rate_hz": 125,
        "channel_order": [
            "ecg_z", "ppg_z", "ecg_valid", "ppg_valid"
        ],
        "ecg_preference": ecg_preference,
        "batch_size": batch_size,
    }
    policy_path = test_dir / "policy.json"
    if policy_path.exists():
        assert json.loads(policy_path.read_text()) == policy
    else:
        policy_path.write_text(
            json.dumps(policy, indent=2), encoding="utf-8"
        )

    taps = firwin(
        41, cutoff=0.5, window=("kaiser", 5.0)
    )

    for batch_number, start in enumerate(
        range(0, len(test_rows), batch_size)
    ):
        batch = test_rows.iloc[start:start + batch_size]
        shard = test_dir / f"batch_{batch_number:03d}.npz"
        receipt = test_dir / f"batch_{batch_number:03d}.json"
        expected_ids = batch["event"].tolist()

        if shard.exists() and receipt.exists():
            saved = json.loads(receipt.read_text())
            assert saved["source_id"] == source_id
            assert saved["event_ids"] == expected_ids
            assert sha256_file(shard) == saved["shard_sha256"]
            print(
                f"Reused verified batch {batch_number + 1}/10.",
                flush=True,
            )
            continue

        signals, masks, summaries = [], [], []

        for row in batch.itertuples(index=False):
            assert Path(row.record).name == row.record
            assert Path(row.event).name == row.event

            header_member = (
                f"{archive_root}/annotated_waveforms/"
                f"{row.record}/{row.event}.hea"
            )
            assert header_member in name_set

            with tempfile.TemporaryDirectory(
                prefix="vtac-test-", dir="/content"
            ) as temporary:
                folder = Path(temporary)
                base = folder / row.event
                base.with_suffix(".hea").write_bytes(
                    verified_read(header_member)
                )
                header = wfdb.rdheader(str(base))
                assert header.record_name == row.event
                assert header.fs == 250
                assert header.sig_len == 90000
                assert len(header.sig_name) == len(set(header.sig_name))

                parent = PurePosixPath(header_member).parent
                for filename in sorted(set(header.file_name)):
                    assert PurePosixPath(filename).name == filename
                    assert "/" not in filename and "\\" not in filename
                    member = str(parent / filename)
                    assert member in name_set
                    (folder / filename).write_bytes(
                        verified_read(member)
                    )

                waveform = wfdb.rdrecord(
                    str(base),
                    sampfrom=72500,
                    sampto=75000,
                    physical=True,
                )
                values = waveform.p_signal
                assert values.shape == (2500, header.n_sig)

                ecg_name = None
                for candidate in ecg_preference:
                    if candidate not in waveform.sig_name:
                        continue
                    index = waveform.sig_name.index(candidate)
                    if np.isfinite(values[:, index]).any():
                        ecg_name = candidate
                        break

                selected = [
                    ecg_name,
                    "PLETH" if "PLETH" in waveform.sig_name else None,
                ]
                x = np.zeros((2, 2500), dtype=np.float32)
                valid = np.zeros((2, 2500), dtype=bool)

                for channel, name in enumerate(selected):
                    if name is None:
                        continue
                    index = waveform.sig_name.index(name)
                    if channel == 0:
                        assert waveform.units[index] == "mV"
                    channel_values = values[:, index]
                    valid[channel] = np.isfinite(channel_values)
                    x[channel, valid[channel]] = (
                        channel_values[valid[channel]]
                    )

                assert np.isfinite(x).all()
                signals.append(x)
                masks.append(valid)
                summaries.append({
                    "event_id": row.event,
                    "record_id": row.record,
                    "true_vt": int(row.decision == "True"),
                    "selected_ecg": ecg_name,
                    "ecg_native_valid_samples": int(valid[0].sum()),
                    "ppg_native_valid_samples": int(valid[1].sum()),
                })

            completed = start + len(signals)
            if completed % 25 == 0:
                elapsed = time.monotonic() - started
                eta = elapsed / completed * (
                    len(test_rows) - completed
                )
                print(
                    f"{completed}/482 events | "
                    f"elapsed {elapsed/60:.1f} min | "
                    f"ETA {eta/60:.1f} min",
                    flush=True,
                )

        # Same preparation as notebook 01, Cell 18.
        native = np.stack(signals).astype(np.float64)
        valid = np.stack(masks)
        count = valid.sum(axis=-1, keepdims=True)
        denominator = np.maximum(count, 1)
        mean = np.where(valid, native, 0).sum(
            axis=-1, keepdims=True
        ) / denominator
        centred = np.where(valid, native - mean, 0)
        variance = (centred ** 2).sum(
            axis=-1, keepdims=True
        ) / denominator
        scale = np.maximum(np.sqrt(variance), 1e-6)
        normalised = centred / scale

        downsampled = resample_poly(
            normalised,
            up=1,
            down=2,
            axis=-1,
            window=taps,
            padtype="constant",
        )
        safe_native = minimum_filter1d(
            valid.astype(np.uint8),
            size=41,
            axis=-1,
            mode="constant",
            cval=0,
        ).astype(bool)
        safe_125 = safe_native[..., ::2]
        assert downsampled.shape == safe_125.shape

        downsampled = np.where(
            safe_125, downsampled, 0
        )
        prepared = np.concatenate(
            [downsampled, safe_125.astype(np.float64)],
            axis=1,
        ).astype(np.float32)
        assert prepared.shape == (len(batch), 4, 1250)
        assert np.isfinite(prepared).all()

        partial = shard.with_suffix(".partial")
        with partial.open("wb") as stream:
            np.savez_compressed(
                stream,
                X=prepared,
                y=batch["decision"].eq("True").to_numpy(
                    dtype=np.int64
                ),
                event_id=batch["event"].to_numpy(dtype=str),
                record_id=batch["record"].to_numpy(dtype=str),
                split=np.repeat("test", len(batch)),
            )
        partial.replace(shard)

        batch_receipt = {
            "source_id": source_id,
            "event_ids": expected_ids,
            "shard_sha256": sha256_file(shard),
            "event_summaries": summaries,
        }
        receipt_partial = receipt.with_suffix(".partial")
        receipt_partial.write_text(
            json.dumps(batch_receipt, indent=2),
            encoding="utf-8",
        )
        receipt_partial.replace(receipt)
        print(
            f"Saved verified batch {batch_number + 1}/10.",
            flush=True,
        )

# Assemble verified batches into one test input file.
parts = {key: [] for key in (
    "X", "y", "event_id", "record_id", "split"
)}
quality_rows = []

for batch_number, start in enumerate(
    range(0, len(test_rows), batch_size)
):
    shard = test_dir / f"batch_{batch_number:03d}.npz"
    receipt = test_dir / f"batch_{batch_number:03d}.json"
    saved = json.loads(receipt.read_text())
    assert sha256_file(shard) == saved["shard_sha256"]

    with np.load(shard, allow_pickle=False) as data:
        for key in parts:
            parts[key].append(data[key].copy())
    quality_rows.extend(saved["event_summaries"])

test_data = {
    key: np.concatenate(chunks)
    for key, chunks in parts.items()
}
assert test_data["X"].shape == (482, 4, 1250)
assert test_data["event_id"].tolist() == (
    test_rows["event"].tolist()
)
assert test_data["record_id"].tolist() == (
    test_rows["record"].tolist()
)
assert test_data["y"].tolist() == (
    test_rows["decision"].eq("True").astype(int).tolist()
)
assert np.all(test_data["split"] == "test")
assert np.isfinite(test_data["X"]).all()

test_path = test_dir / "test.npz"
checksum_path = test_dir / "test.sha256"

if test_path.exists() and checksum_path.exists():
    assert sha256_file(test_path) == checksum_path.read_text().strip()
else:
    partial = test_dir / "test.partial"
    with partial.open("wb") as stream:
        np.savez_compressed(stream, **test_data)
    partial.replace(test_path)
    checksum_path.write_text(
        sha256_file(test_path), encoding="utf-8"
    )

pd.DataFrame(quality_rows).to_csv(
    test_dir / "test_signal_quality_manifest.csv",
    index=False,
)

print("\nTEST INPUT PREPARATION COMPLETE")
print("Events:", len(test_data["y"]))
print("True VT:", int(test_data["y"].sum()))
print("Input shape:", test_data["X"].shape)
print(
    "Usable ECG:",
    int(test_data["X"][:, 2, :].any(axis=1).sum()),
)
print(
    "Usable PPG:",
    int(test_data["X"][:, 3, :].any(axis=1).sum()),
)
print("Frozen models:", len(frozen_models))
print("Model scoring performed: 0")
print("Prepared test:", test_path)
print("Frozen plan:", frozen_path)
print(f"Completed in {(time.monotonic() - started)/60:.1f} min.")

In [ ]:
# M2 CELL 6 — one frozen test evaluation of all five models.
# ETA: ~5–30 seconds. No fitting or threshold selection.

import hashlib, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)

started = time.monotonic()

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()

frozen = json.loads((run_dir / "frozen_test_plan.json").read_text())
test_dir = RUN_DIR / "vtac_m2_test_prepared_v1"
test_path = test_dir / "test.npz"

assert file_sha256(test_path) == (
    test_dir / "test.sha256"
).read_text().strip()
assert frozen["preprocessing_sha256"] == file_sha256(
    PREP_DIR / "preprocessing.json"
)
assert frozen["development_npz_sha256"] == (
    PREP_DIR / "development.sha256"
).read_text().strip()

expected_seeds = [
    20261015, 20261016, 20261017, 20261018, 20261019
]
assert [item["seed"] for item in frozen["models"]] == expected_seeds

with np.load(test_path, allow_pickle=False) as saved:
    X_test = saved["X"].copy()
    y_test = saved["y"].astype(int).copy()
    test_events = saved["event_id"].astype(str).copy()
    test_records = saved["record_id"].astype(str).copy()
    test_splits = saved["split"].astype(str).copy()

assert X_test.shape == (482, 4, 1250)
assert y_test.shape == (482,)
assert int(y_test.sum()) == 137
assert len(set(test_events.tolist())) == 482
assert len(set(test_records.tolist())) == 226
assert np.all(test_splits == "test")
assert not set(test_records) & set(records.astype(str))
assert np.isfinite(X_test).all()
assert np.isin(X_test[:, 2:, :], [0, 1]).all()
assert (X_test[:, :2, :][X_test[:, 2:, :] == 0] == 0).all()

ppg_available = X_test[:, 3, :].any(axis=1)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_test).float()),
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

def test_metrics(truth, probability, prediction):
    tn, fp, fn, tp = confusion_matrix(
        truth, prediction, labels=[0, 1]
    ).ravel()
    both_classes = len(np.unique(truth)) == 2

    return {
        "n": int(len(truth)),
        "true_vt": int(truth.sum()),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "recall_true_vt": float(
            recall_score(truth, prediction, zero_division=0)
        ),
        "specificity_false_alarm": float(
            tn / (tn + fp) if tn + fp else float("nan")
        ),
        "precision_true_vt": float(
            precision_score(truth, prediction, zero_division=0)
        ),
        "f1_true_vt": float(
            f1_score(truth, prediction, zero_division=0)
        ),
        "auroc": float(roc_auc_score(truth, probability))
            if both_classes else float("nan"),
        "average_precision": float(
            average_precision_score(truth, probability)
        ) if both_classes else float("nan"),
    }

evaluation_dir = run_dir / "frozen_test_evaluation"
evaluation_dir.mkdir(exist_ok=True)

evaluation_manifest = {
    "test_input_sha256": file_sha256(test_path),
    "frozen_plan_sha256": file_sha256(
        run_dir / "frozen_test_plan.json"
    ),
    "primary_predeclared_seed": 20261015,
    "all_five_seeds_reported": expected_seeds,
    "no_test_fitting_or_threshold_selection": True,
}
manifest_path = evaluation_dir / "evaluation_manifest.json"
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == evaluation_manifest
else:
    manifest_path.write_text(
        json.dumps(evaluation_manifest, indent=2),
        encoding="utf-8",
    )

rows = []
subgroup_rows = []

print("Scoring five frozen models — ETA ~5–30 seconds.", flush=True)
for number, item in enumerate(frozen["models"], start=1):
    checkpoint_path = Path(item["checkpoint"])
    assert file_sha256(checkpoint_path) == (
        item["checkpoint_sha256"]
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    current_model = CompactM2FCN().to(device)
    current_model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True,
    )
    current_model.eval()

    chunks = []
    with torch.no_grad():
        for (batch_x,) in test_loader:
            batch_x = batch_x.to(
                device, non_blocking=(device.type == "cuda")
            )
            with torch.amp.autocast(
                "cuda", enabled=(device.type == "cuda")
            ):
                logits = current_model(batch_x)
            chunks.append(
                torch.sigmoid(logits).float().cpu().numpy()
            )

    probability = np.concatenate(chunks)
    assert probability.shape == (482,)
    assert np.isfinite(probability).all()
    assert ((probability >= 0) & (probability <= 1)).all()

    threshold = float(item["threshold"])
    prediction = (probability >= threshold).astype(int)

    overall = test_metrics(y_test, probability, prediction)
    overall.update({
        "seed": int(item["seed"]),
        "best_epoch": int(item["best_epoch"]),
        "threshold": threshold,
    })
    rows.append(overall)

    for scope, mask in (
        ("ppg_available", ppg_available),
        ("ppg_unavailable", ~ppg_available),
    ):
        subgroup = test_metrics(
            y_test[mask],
            probability[mask],
            prediction[mask],
        )
        subgroup.update({
            "seed": int(item["seed"]),
            "scope": scope,
        })
        subgroup_rows.append(subgroup)

    prediction_path = (
        evaluation_dir
        / f"test_predictions_seed{item['seed']}.csv"
    )
    pd.DataFrame({
        "event_id": test_events,
        "record_id": test_records,
        "target_true_vt": y_test,
        "probability_true_vt": probability,
        "threshold": threshold,
        "prediction": prediction,
        "ppg_available": ppg_available,
    }).to_csv(prediction_path, index=False)

    elapsed = time.monotonic() - started
    eta = elapsed / number * (5 - number)
    print(
        f"Seed {number}/5 scored | elapsed {elapsed:.1f}s | "
        f"ETA {eta:.1f}s",
        flush=True,
    )

results = pd.DataFrame(rows).sort_values("seed")
subgroups = pd.DataFrame(subgroup_rows).sort_values(
    ["scope", "seed"]
)
metric_columns = [
    "recall_true_vt",
    "specificity_false_alarm",
    "precision_true_vt",
    "f1_true_vt",
    "auroc",
    "average_precision",
]
summary = results[metric_columns].agg(["mean", "std"])

results.to_csv(
    evaluation_dir / "five_seed_test_metrics.csv",
    index=False,
)
subgroups.to_csv(
    evaluation_dir / "five_seed_test_ppg_subgroups.csv",
    index=False,
)
summary.to_csv(
    evaluation_dir / "five_seed_test_summary.csv"
)

print("\nFROZEN TEST RESULTS")
print(results[
    ["seed", "n", "true_vt", "TN", "FP", "FN", "TP"]
    + metric_columns
].round(4).to_string(index=False))

print("\nMEAN AND SAMPLE SD ACROSS FIVE SEEDS")
print(summary.round(4).to_string())

print("\nPPG AVAILABILITY SUBGROUPS")
print(subgroups.groupby("scope")[
    ["n", "true_vt", "recall_true_vt",
     "specificity_false_alarm", "average_precision"]
].mean().round(4).to_string())

print("\nSaved:", evaluation_dir)
print("No fitting or threshold selection used test data.")
print(
    "Seed SD measures training variation, "
    "not patient-level uncertainty."
)
print(f"Completed in {time.monotonic() - started:.1f}s.")